In [1]:
!pip install ultralytics roboflow -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.3/260.3 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 87.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
httpx2 2.4.0 requires idna>=3.18, but you have idna 3.7 which is incompatible.


In [2]:
from roboflow import Roboflow
rf = Roboflow(api_key="key")
project = rf.workspace("defectdatasets").project("neu-det-fquva")
dataset = project.version(1).download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to NEU-DET-1 in yolov8:: 100%|██████████| 3610/3610 [00:00<00:00, 11190.81it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
YAML = """
path: /content/NEU-DET-1
train: train/images
val:   valid/images
test:  test/images
nc: 6
names:
  - crazing
  - inclusion
  - patches
  - pitted_surface
  - rolled-in_scale
  - scratches
"""
with open("/content/neu_det.yaml", "w") as f:
    f.write(YAML)
print("✅ 완료")

✅ 완료


In [4]:
from ultralytics import YOLO
import json

MODELS = {
    "YOLOv8s":  "yolov8s.pt",
    "YOLOv11s": "yolo11s.pt",
    "YOLOv26s": "yolo26s.pt",
}
CLASS_NAMES = ["crazing","inclusion","patches",
               "pitted_surface","rolled-in_scale","scratches"]
YAML_PATH = "/content/neu_det.yaml"
results = {}

for name, weight in MODELS.items():
    print(f"\n{'='*40}\n  {name} 학습 중...\n{'='*40}")
    model = YOLO(weight)
    model.train(
        data=YAML_PATH, epochs=50, imgsz=640,
        batch=16, device=0, verbose=False,
        project="/content/runs", name=name, exist_ok=True,
    )
    metrics = YOLO(f"/content/runs/{name}/weights/best.pt").val(
        data=YAML_PATH, imgsz=640, device=0, verbose=False,
    )
    per_class = {}
    for i, cls in enumerate(CLASS_NAMES):
        try: per_class[cls] = round(float(metrics.box.ap[i]), 4)
        except: per_class[cls] = 0.0

    results[name] = {
        "mAP50":     round(float(metrics.box.map50), 4),
        "mAP50_95":  round(float(metrics.box.map),   4),
        "precision": round(float(metrics.box.mp),    4),
        "recall":    round(float(metrics.box.mr),    4),
        "per_class": per_class,
    }
    print(f"  mAP@50: {results[name]['mAP50']}")

with open("/content/map_small.json", "w") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print("\n✅ 완료")


  YOLOv8s 학습 중...
Ultralytics 8.4.84 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/neu_det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=YOLOv8s, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True,

In [5]:
from google.colab import files
files.download("/content/map_small.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>